# Anomaly Detection for Cyber Threats

**Section 3 of the ML in Cyber Security assignment (25 marks)**

This notebook builds and compares two unsupervised/semi-supervised models for detecting anomalous (attack) network traffic without relying on attack labels during training, and is designed to run in **Google Colab**: you upload the dataset CSV yourself when prompted, no Kaggle API or credential setup needed.

1. **Classic Machine Learning:** Isolation Forest, trained only on normal traffic
2. **Deep Learning:** Autoencoder Neural Network, trained only on normal traffic, using reconstruction error as the anomaly score

**Dataset:** [CIC-IDS2018](https://www.unb.ca/cic/datasets/ids-2018.html) (Canadian Institute for Cybersecurity), specifically the `Wednesday-14-02-2018` capture: 1,048,575 network flows summarized by CICFlowMeter into numeric features (packet counts/sizes, timing, flags, etc.), a `Protocol` field, a `Timestamp`, and a `Label`. This particular day contains 3 label values: `Benign`, `FTP-BruteForce`, and `SSH-Bruteforce`. For this anomaly-detection task, `FTP-BruteForce` and `SSH-Bruteforce` are both treated as the single `Anomaly` class, with `Benign` as `Normal` - the models never see the original 3-way split, only normal vs anomaly, and in fact never see any anomalies at all during training (see below).

**Both models are trained exclusively on normal traffic.** This is the standard semi-supervised anomaly-detection setup: each model learns what "normal" looks like from benign flows only, then flags anything that deviates enough from that pattern at test time - including attack types the model was never shown, which is the intended proxy for detecting unknown/zero-day threats.

**Before running:** have `cic.csv` (about 358MB) ready on your computer. You do **not** need to upload it into Colab yourself first - the data-loading cell below will open Colab's upload dialog automatically if it can't find the file. See the "Load Dataset" section below for exactly where to put it if you'd rather upload ahead of time via the Files sidebar.

In [ ]:
import os

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Running in Google Colab:", IN_COLAB)
print("Working directory:", os.getcwd())

In [ ]:
import gc
import glob
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    precision_score, recall_score, f1_score, classification_report,
    confusion_matrix, roc_curve, auc, precision_recall_curve, average_precision_score,
)

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping

import joblib

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

OUTPUT_DIR = "outputs"
FIGURES_DIR = os.path.join(OUTPUT_DIR, "figures")
MODELS_DIR = os.path.join(OUTPUT_DIR, "models")
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print("Setup complete.")

## 1. Load Dataset (Google Colab Upload)

This notebook does **not** download anything itself. It looks for `cic.csv` in the most common places a file ends up after uploading it in Colab:

- `data/cic.csv` (if you created a `data` folder in the Files sidebar and uploaded into it)
- `cic.csv` directly in `/content` (the default location if you just drag-and-drop the file into the Files sidebar root)

**If it can't find the file anywhere, the cell below will automatically open Colab's upload dialog** so you can pick the CSV from your computer - no need to fix folders/paths yourself.

Note: anything uploaded this way only lives for the current Colab session/runtime - if the runtime restarts or disconnects, you'll need to upload again. At about 358MB, this may take a few minutes over a browser upload.

In [ ]:
EXPECTED_FILENAME = "cic.csv"
CANDIDATE_DIRS = ["data", ".", "/content", "/content/data"]


def find_dataset_path():
    checked_dirs = []
    for directory in CANDIDATE_DIRS:
        if directory in checked_dirs:
            continue
        checked_dirs.append(directory)
        exact_path = os.path.join(directory, EXPECTED_FILENAME)
        if os.path.exists(exact_path):
            return exact_path

    for directory in checked_dirs:
        if os.path.isdir(directory):
            csv_files = sorted(glob.glob(os.path.join(directory, "*.csv")))
            if len(csv_files) == 1:
                return csv_files[0]

    return None


def load_cic_ids2018():
    dataset_path = find_dataset_path()

    if dataset_path is None and IN_COLAB:
        print(
            f"Could not find '{EXPECTED_FILENAME}' in {CANDIDATE_DIRS}. "
            "Opening the upload dialog -- please choose the CSV file from your computer..."
        )
        from google.colab import files
        files.upload()  # uploaded file(s) are saved into the current working directory
        dataset_path = find_dataset_path()

    if dataset_path is None:
        raise FileNotFoundError(
            f"Could not find the dataset. Expected '{EXPECTED_FILENAME}' in one of: {CANDIDATE_DIRS}.\n"
            "In Google Colab: re-run this cell to trigger the upload dialog, or manually upload the "
            "CSV via the Files sidebar (into '/content' or a 'data' subfolder), then re-run this cell.\n"
            "See README.md for details."
        )

    print(f"Loading dataset from '{dataset_path}' ...")
    return pd.read_csv(dataset_path)


df = load_cic_ids2018()
print("Shape:", df.shape)
df.head()

## 2. Data Cleaning

Following the assignment's preprocessing steps for this section (remove duplicates, handle missing values):

1. **`Timestamp` is dropped.** It is a raw date/time string, not a numeric traffic feature, and is not used as a predictor.
2. **Exact duplicate rows are removed.** Checking the raw file directly showed about 21.5% of rows are exact duplicates (225,628 out of 1,048,575) - concentrated much more heavily in the attack traffic (about 80% of `FTP-BruteForce` rows and about 37% of `SSH-Bruteforce` rows are duplicates, versus under 1% of `Benign` rows). This makes sense: automated brute-force login attempts tend to generate near-identical flow patterns each time, unlike organic benign traffic.
3. **Infinite and missing values are removed.** `Flow Byts/s` and `Flow Pkts/s` contain literal `Infinity`/`NaN` values for zero-duration flows, same as in the CIC-IDS2017 data used in Section 2.
4. **The 3-class `Label` is collapsed to a binary target:** `Normal` (`Benign`) vs `Anomaly` (`FTP-BruteForce` or `SSH-Bruteforce` combined). The models below never see the original attack-type labels, only this binary distinction, and during training they never see the `Anomaly` class at all (see Section 5).

In [ ]:
print("Rows before cleaning:", len(df))
print("Distinct labels:", df["Label"].unique())

df = df.drop(columns=["Timestamp"])

n_before_dedup = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {n_before_dedup - len(df)} duplicate rows ({n_before_dedup} -> {len(df)}).")

feature_cols = [c for c in df.columns if c != "Label"]
df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors="coerce")
df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], np.nan)

n_before_na = len(df)
df = df.dropna(subset=feature_cols).reset_index(drop=True)
print(f"Removed {n_before_na - len(df)} rows with missing/infinite values ({n_before_na} -> {len(df)}).")

df[feature_cols] = df[feature_cols].astype("float32")
gc.collect()

df["is_anomaly"] = (df["Label"] != "Benign").astype(int)

print("\nOriginal label distribution after cleaning:")
print(df["Label"].value_counts())
print("\nBinary label distribution (0 = Normal, 1 = Anomaly):")
print(df["is_anomaly"].value_counts())

## 3. Exploratory Data Analysis

A look at the binary class balance and a couple of the network traffic characteristics the assignment specifically calls out (packet size, protocol type, connection duration).

In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(x="is_anomaly", data=df)
plt.xticks([0, 1], ["Normal (0)", "Anomaly (1)"])
plt.title("Binary Class Distribution")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "class_distribution.png"), dpi=150)
plt.show()

plt.figure(figsize=(6, 4))
sns.countplot(x="Protocol", hue="is_anomaly", data=df)
plt.title("Protocol Type by Class")
plt.xlabel("Protocol (6 = TCP, 17 = UDP, 0 = other)")
plt.legend(title="is_anomaly", labels=["Normal", "Anomaly"])
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "protocol_by_class.png"), dpi=150)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(x="is_anomaly", y="Flow Duration", data=df, ax=axes[0], showfliers=False)
axes[0].set_xticklabels(["Normal", "Anomaly"])
axes[0].set_title("Connection Duration by Class")

sns.boxplot(x="is_anomaly", y="Pkt Size Avg", data=df, ax=axes[1], showfliers=False)
axes[1].set_xticklabels(["Normal", "Anomaly"])
axes[1].set_title("Average Packet Size by Class")

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "duration_and_packet_size_by_class.png"), dpi=150)
plt.show()

## 4. Feature Selection and Preparation

Rather than manually narrowing down to only three raw columns, the full set of CICFlowMeter-engineered numeric flow features is used as the feature space - it already includes multiple columns representing each characteristic the assignment names as an example (packet size: `Pkt Len Mean/Min/Max/Std`, `Pkt Size Avg`, etc.; protocol type: `Protocol`; connection duration: `Flow Duration`, IAT features), plus many others describing the same underlying traffic behaviour from different angles. `Protocol` is categorical (a small numeric code, not a magnitude), so it is one-hot encoded rather than left as a raw number a distance-based model would otherwise treat as continuous.

In [ ]:
protocol_dummies = pd.get_dummies(df["Protocol"], prefix="Protocol")
model_feature_cols = [c for c in feature_cols if c != "Protocol"]

X = pd.concat([df[model_feature_cols], protocol_dummies], axis=1).to_numpy(dtype="float32")
y = df["is_anomaly"].to_numpy()

print("Feature matrix shape:", X.shape)
print("Protocol categories one-hot encoded:", list(protocol_dummies.columns))

del protocol_dummies
gc.collect()

## 5. Train/Test Split and Normalization

An 80/20 stratified split keeps the normal/anomaly ratio the same in both sets. Both models are then fit **only on the normal (`is_anomaly == 0`) rows of the training split** - this is what makes them semi-supervised anomaly detectors rather than ordinary binary classifiers: they learn a model of normal traffic only, never seeing an actual attack example, and are evaluated on the full test set (which does contain both classes). `StandardScaler` is fit on that normal-only training subset only, then applied to everything else, so no information about the test set (or about anomalies at all) leaks into preprocessing.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
del X
gc.collect()

X_train_normal = X_train[y_train == 0]
print(f"Train size: {len(X_train)} (normal-only subset used for fitting: {len(X_train_normal)})")
print(f"Test size: {len(X_test)} (Normal: {(y_test == 0).sum()}, Anomaly: {(y_test == 1).sum()})")

scaler = StandardScaler()
X_train_normal_scaled = scaler.fit_transform(X_train_normal)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, os.path.join(MODELS_DIR, "scaler.joblib"))

del X_train, X_train_normal
gc.collect()

## 6. Evaluation Helper

Both models below produce a continuous anomaly score per flow (higher = more anomalous). A single shared function turns that into everything the assignment asks for:

- A hard anomaly/normal decision using a threshold set from the **95th percentile of the anomaly score on the normal-only training data** - i.e. "flag anything scored higher than 95% of known-normal traffic," a standard way to pick an operating point without ever looking at attack examples.
- **TPR** (recall on the anomaly class) and **FPR**, computed at that threshold.
- A confusion matrix, an ROC curve (TPR vs FPR across all thresholds), and a precision-recall curve.

In [ ]:
def evaluate_anomaly_detector(y_true, scores, threshold, model_name, filename_prefix):
    y_pred = (scores >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"=== {model_name} (threshold = {threshold:.4f}) ===")
    print(f"TPR (True Positive Rate / Recall on anomalies): {tpr:.4f}")
    print(f"FPR (False Positive Rate): {fpr:.4f}")
    print(classification_report(y_true, y_pred, target_names=["Normal", "Anomaly"], zero_division=0))

    plt.figure(figsize=(5, 4))
    sns.heatmap(
        confusion_matrix(y_true, y_pred, labels=[0, 1]), annot=True, fmt="d", cmap="Blues",
        xticklabels=["Normal", "Anomaly"], yticklabels=["Normal", "Anomaly"],
    )
    plt.title(f"{model_name} - Confusion Matrix")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, f"{filename_prefix}_confusion_matrix.png"), dpi=150)
    plt.show()

    fpr_curve, tpr_curve, _ = roc_curve(y_true, scores)
    roc_auc = auc(fpr_curve, tpr_curve)
    plt.figure(figsize=(5, 4))
    plt.plot(fpr_curve, tpr_curve, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--", color="grey")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{model_name} - ROC Curve")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, f"{filename_prefix}_roc_curve.png"), dpi=150)
    plt.show()

    prec_curve, rec_curve, _ = precision_recall_curve(y_true, scores)
    ap = average_precision_score(y_true, scores)
    plt.figure(figsize=(5, 4))
    plt.plot(rec_curve, prec_curve, label=f"AP = {ap:.3f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"{model_name} - Precision-Recall Curve")
    plt.legend(loc="lower left")
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, f"{filename_prefix}_precision_recall_curve.png"), dpi=150)
    plt.show()

    return {
        "TPR": tpr, "FPR": fpr, "Precision": precision, "Recall": recall,
        "F1-score": f1, "ROC-AUC": roc_auc, "Average Precision": ap,
    }

## 7. Classic Machine Learning - Isolation Forest

`IsolationForest` is fit only on the normal-only training subset. Its `score_samples` output is higher for "easy to isolate" (i.e. more normal-looking) points and lower for outliers, so it is negated below to get an anomaly score where higher always means more anomalous, matching the convention used for the autoencoder's reconstruction error later.

In [ ]:
iso_forest = IsolationForest(
    n_estimators=100,
    random_state=SEED,
    n_jobs=-1,
)
iso_forest.fit(X_train_normal_scaled)

train_scores_if = -iso_forest.score_samples(X_train_normal_scaled)
test_scores_if = -iso_forest.score_samples(X_test_scaled)

threshold_if = np.percentile(train_scores_if, 95)
print(f"Isolation Forest threshold (95th percentile of normal training scores): {threshold_if:.4f}")

results_if = evaluate_anomaly_detector(
    y_test, test_scores_if, threshold_if, "Isolation Forest", "isoforest"
)

joblib.dump(iso_forest, os.path.join(MODELS_DIR, "isolation_forest.joblib"))
print("Saved Isolation Forest model to", MODELS_DIR)

## 8. Deep Learning - Autoencoder

A symmetric dense autoencoder is trained to reconstruct normal traffic only, using mean squared error loss. Once trained, it should reconstruct normal flows well (low error) and struggle to reconstruct genuinely different (anomalous) flows (higher error), so the per-row reconstruction error itself becomes the anomaly score.

In [ ]:
INPUT_DIM = X_train_normal_scaled.shape[1]

input_layer = Input(shape=(INPUT_DIM,))
encoded = Dense(64, activation="relu")(input_layer)
encoded = Dense(32, activation="relu")(encoded)
bottleneck = Dense(16, activation="relu")(encoded)
decoded = Dense(32, activation="relu")(bottleneck)
decoded = Dense(64, activation="relu")(decoded)
output_layer = Dense(INPUT_DIM, activation="linear")(decoded)

autoencoder = Model(inputs=input_layer, outputs=output_layer)
autoencoder.compile(optimizer="adam", loss="mse")
autoencoder.summary()

early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

history = autoencoder.fit(
    X_train_normal_scaled, X_train_normal_scaled,
    validation_split=0.1,
    epochs=30,
    batch_size=512,
    callbacks=[early_stop],
    verbose=1,
)

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Reconstruction MSE")
plt.title("Autoencoder Training Loss")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "autoencoder_training_curve.png"), dpi=150)
plt.show()


def reconstruction_error(model, X):
    reconstructed = model.predict(X, verbose=0)
    return np.mean(np.square(X - reconstructed), axis=1)


train_scores_ae = reconstruction_error(autoencoder, X_train_normal_scaled)
test_scores_ae = reconstruction_error(autoencoder, X_test_scaled)

threshold_ae = np.percentile(train_scores_ae, 95)
print(f"Autoencoder threshold (95th percentile of normal training reconstruction error): {threshold_ae:.4f}")

results_ae = evaluate_anomaly_detector(
    y_test, test_scores_ae, threshold_ae, "Autoencoder", "autoencoder"
)

autoencoder.save(os.path.join(MODELS_DIR, "autoencoder.keras"))
print("Saved Autoencoder model to", MODELS_DIR)

## 9. Model Comparison

TPR, FPR, precision, recall, F1-score (all at each model's own 95th-percentile threshold), plus threshold-independent ROC-AUC and average precision, side by side.

In [ ]:
results = pd.DataFrame([
    {"Model": "Isolation Forest", **results_if},
    {"Model": "Autoencoder", **results_ae},
]).set_index("Model")

display(results)

results.plot(kind="bar", figsize=(10, 5), ylim=(0, 1))
plt.title("Model Comparison: Isolation Forest vs Autoencoder")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "model_comparison.png"), dpi=150)
plt.show()

results.to_csv(os.path.join(OUTPUT_DIR, "model_comparison.csv"))
print("Saved comparison table to", os.path.join(OUTPUT_DIR, "model_comparison.csv"))

## 10. Discussion

**Findings.** Fill this in after running the notebook with the actual numbers from the comparison table above - e.g. which model achieved a better TPR/FPR trade-off and higher ROC-AUC, and whether the deep learning model's extra complexity translated into a meaningful gain over the Isolation Forest baseline.

**Cybersecurity implications.** Anomaly detection is specifically valuable for catching attack types a system has never been trained on ("zero-day" behaviour), which is exactly why both models here are trained only on normal traffic rather than on labelled attacks. The threshold choice directly trades off TPR against FPR: a lower threshold catches more real attacks (higher TPR) but also raises more false alarms on benign traffic (higher FPR), which matters operationally since a security team that gets flooded with false positives will start ignoring alerts altogether. The ROC and precision-recall curves let a deployment choose a threshold deliberately instead of accepting the arbitrary 95th-percentile default used here.

**Limitations.**
- This dataset only contains two specific attack types (`FTP-BruteForce` and `SSH-Bruteforce`), both from a single day of capture. Good performance here does not necessarily generalize to genuinely novel attack types, different protocols, or different network environments; it demonstrates the *method*, not a validated production-ready detector.
- The 95th-percentile threshold is a reasonable, simple default, not a tuned operating point. A real deployment would choose it based on the ROC/precision-recall curves and the actual cost of false positives versus false negatives in that environment.
- Exact-duplicate-row removal (this notebook's Data Cleaning step) only catches identical flows; near-duplicate attack traffic (e.g. the same brute-force pattern with minor timing variation) is not caught by this and could still be somewhat over-represented.
- No hyperparameter tuning (e.g. Isolation Forest tree count/contamination, autoencoder depth/bottleneck size/learning rate) was performed; both models use reasonable defaults.
- Using only normal data to train means the models never learn what makes `FTP-BruteForce` different from `SSH-Bruteforce`; that distinction is intentionally invisible to them, since the goal here is normal-vs-anomaly, not attack-type classification (that is the assignment's Section 2, Cyber Attack Detection, on a different dataset).

**Potential improvements.**
- Tune the anomaly-score threshold directly from the ROC curve to hit a specific target FPR (e.g. the highest TPR achievable at FPR under 1%), rather than using a fixed percentile.
- Try a smaller or larger autoencoder bottleneck to see how compression size affects sensitivity to different anomaly types.
- Evaluate on a different CIC-IDS2018 day (with different attack types) to test whether a model trained on this day's normal traffic generalizes to genuinely unseen attack behaviour, which is a closer test of true zero-day detection than evaluating on attacks from the same day.
- Compare against a One-Class SVM (the other classic method the assignment allows) to see whether it agrees with the Isolation Forest's ranking of which flows are most anomalous.